In [1]:
import os
from google.colab import userdata


# Définissez vos informations de manière sécurisée
GITHUB_USER = "koubra-gaby"
#GITHUB_TOKEN = userdata.get('GH_TOKEN')
REPO_NAME = "Chadian_arabic-update"

# Configuration de l'URL avec authentification
REPO_URL = f"https://@github.com/{GITHUB_USER}/{REPO_NAME}.git"

# Cloner le dépôt si ce n'est pas déjà fait
if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
    print("Dépôt cloné avec succès !")
else:
    print("Le dépôt existe déjà.")


Cloning into 'Chadian_arabic-update'...
remote: Enumerating objects: 239, done.
remote: Counting objects: 100% (239/239), done.
remote: Compressing objects: 100% (169/169), done.
remote: Total 239 (delta 103), reused 194 (delta 64), pack-reused 0 (from 0)
Receiving objects: 100% (239/239), 6.41 MiB | 9.34 MiB/s, done.
Resolving deltas: 100% (103/103), done.
Dépôt cloné avec succès !


In [ ]:
# via python & Jupyter
from huggingface_hub import notebook_login
notebook_login()

### Entrainer un nouveau model sentencepiece et expandre le tokenizer de nllb

In [ ]:
import sentencepiece as spm

# Chemin vers votre fichier de texte brut (ex: corpus.txt)

input_corpus = "/content/Chadian_arabic-update/Data/full_dataset/arabe_tchad3.txt"
model_prefix = "shu_Latn_sp"

# Entraînement du tokenizer
spm.SentencePieceTrainer.train(
    input=input_corpus,
    model_prefix=model_prefix,
    vocab_size=8000,          # Taille cible du vocabulaire pour votre langue
    model_type="BPE",         # NLLB utilise le format BPE (Byte Pair Encoding)
    character_coverage=0.9995,# Couverture des caractères (0.9995 recommandé pour les langues à alphabet)
    byte_fallback=True        # Crucial pour éviter les tokens inconnus (<unk>)
)


True

### Extraire et nettoyer les news tokens

In [ ]:
# Charger les tokens générés par SentencePiece
new_tokens = []
with open(f"{model_prefix}.vocab", "r", encoding="utf-8") as f:
    for line in f:
        token = line.split("\t")[0]
        # On ignore les tokens de contrôle par défaut de SentencePiece
        if token not in ["<unk>", "<s>", "</s>", "<pad>"]:
            new_tokens.append(token)

print(f"Nombre de tokens extraits : {len(new_tokens)}")


Nombre de tokens extraits : 7997


In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_id = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

# Récupérer le vocabulaire actuel de NLLB
nllb_vocab = tokenizer.get_vocab()

# Filtrer les nouveaux tokens pour ne garder que ceux absents de NLLB
tokens_to_add = [t for t in new_tokens if t not in nllb_vocab]

# 1. Ajouter le code de la nouvelle langue (ex: 'new_Latn')
new_lang_code = "shu_Latn"
tokenizer.add_special_tokens({"additional_special_tokens": [new_lang_code]})

# 4. Mettre à jour les dictionnaires internes de NLLB pour associer le token à une langue
# Cela évite les erreurs lors de la configuration de `src_lang` ou `tgt_lang`
tokenizer.added_tokens_encoder[new_lang_code] = len(tokenizer) - 1
tokenizer.added_tokens_decoder[len(tokenizer) - 1] = new_lang_code

# 2. Ajouter les nouveaux tokens textuels issus de SentencePiece
tokenizer.add_tokens(tokens_to_add)

# 3. Redimensionner les embeddings du modèle
model.resize_token_embeddings(len(tokenizer))

# 4. Sauvegarder le tout
tokenizer.save_pretrained("/content/Chadian_arabic-update/expanded_model_and_tokenizer/nllb-random-custom-tokenizer")
model.save_pretrained("/content/Chadian_arabic-update/expanded_model_and_tokenizer/nllb-random-custom-tokenizer")

print(f"Terminé ! {len(tokens_to_add)} nouveaux tokens et la langue '{new_lang_code}' ont été ajoutés.")


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.3MB            

...

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Terminé ! 5950 nouveaux tokens et la langue 'shu_Latn' ont été ajoutés.


In [ ]:
model.push_to_hub("koubra-gaby/nllb-random-custom-tokenizer")
tokenizer.push_to_hub("koubra-gaby/nllb-random-custom-tokenizer")

In [ ]:
! git add .
! git commmit